In [1]:
import pandas as pd


In [2]:

allergen_df = pd.read_csv("../datasets/food_ingredients_and_allergens.csv")
alternatives = pd.read_csv("../datasets/alternative_foods.csv")


In [4]:
print("Allergen dataset:", allergen_df.shape)
print("Alternative dataset:", alternatives.shape)

print("\nAllergen columns:")
print(allergen_df.columns.tolist())

print("\nAlternative columns:")
print(alternatives.columns.tolist())

Allergen dataset: (399, 7)
Alternative dataset: (197, 13)

Allergen columns:
['Food Product', 'Main Ingredient', 'Sweetener', 'Fat/Oil', 'Seasoning', 'Allergens', 'Prediction']

Alternative columns:
['Food', 'food_type', 'consumption_type', 'energy', 'protein', 'carbs', 'fat', 'fiber', 'sugars', 'sodium', 'is_leafy_green', 'is_alcohol', 'quantity']


In [5]:
def check_allergy(food_name, user_allergies):
    food_name_lower = food_name.lower()

    matched_rows = allergen_df[
        allergen_df["Food Product"].astype(str).str.lower().str.contains(food_name_lower, na=False)
    ]

    detected = []

    for _, row in matched_rows.iterrows():
        allergens_text = str(row["Allergens"]).lower()

        for allergy in user_allergies:
            if allergy.lower() in allergens_text:
                detected.append(allergy)

    return list(set(detected))

In [6]:
check_allergy("Almond Cookies", ["Dairy", "Wheat"])

['Wheat', 'Dairy']

In [7]:
def suggest_alternatives(original_energy, max_results=5):
    df = alternatives.copy()

    if "is_alcohol" in df.columns:
        df = df[df["is_alcohol"] == 0]

    df["energy_difference"] = abs(df["energy"] - original_energy)

    sort_cols = ["energy_difference"]
    ascending = [True]

    if "sugars" in df.columns:
        sort_cols.append("sugars")
        ascending.append(True)

    if "sodium" in df.columns:
        sort_cols.append("sodium")
        ascending.append(True)

    df = df.sort_values(by=sort_cols, ascending=ascending)

    available_cols = [
        col for col in ["Food", "food_type", "energy", "protein", "carbs", "fat", "sugars", "sodium"]
        if col in df.columns
    ]

    return df[available_cols].head(max_results)

In [8]:
suggest_alternatives(350)

,Food,food_type,energy,protein,carbs,fat,sugars,sodium
28,Helapa,dessert,300,4,45,12,25,100
29,Wandu,dessert,300,4,45,12,25,100
30,Aggala,dessert,300,4,45,12,25,100
31,Lavariya,dessert,300,4,45,12,25,100
89,Curd (Buffalo) + Treacle,dessert,300,4,45,12,25,100
